# Coupled 0-π qubits

Two symmetric 0-π qubits couple through capacitors $C_c$ to two shared coupler nodes, following [Liu, Weissler, and Combes, arXiv:2609.16630 (2026)](https://doi.org/10.48550/arXiv.2609.16630).
Each qubit node also has a capacitance $C_0$ to ground.
This is scheme (a) of that paper, a pure $n_{\theta_1} n_{\theta_2}$ coupler.

In [ ]:
import sympy as sym
import circuit_quantization as cq
from circuit_quantization import Branch, Circuit, Loops, quantize

## Circuit

Qubit 1 has nodes 1–4 and qubit 2 nodes 5–8; nodes 9 and 10 are the coupler and node 0 is ground.
Nodes 1, 4, 5, 8 couple to node 10, and nodes 2, 3, 6, 7 to node 9.

In [ ]:
C_J, C, L, C_0, C_c, E_J = sym.symbols("C_J C L C_0 C_c E_J", positive=True)

branches = []
for q, (a, b, c, d) in enumerate(((1, 2, 3, 4), (5, 6, 7, 8)), start=1):
    branches += [
        Branch(f"J{q}a", a, b, C=C_J, EJ=E_J), Branch(f"J{q}b", c, d, C=C_J, EJ=E_J),
        Branch(f"L{q}a", b, c, L=L), Branch(f"L{q}b", d, a, L=L),
        Branch(f"C{q}a", a, c, C=C), Branch(f"C{q}b", b, d, C=C),
        *(Branch(f"C0_{n}", n, 0, C=C_0) for n in (a, b, c, d)),
    ]
branches += [Branch(f"Cc_{n}", n, 10, C=C_c) for n in (1, 4, 5, 8)]
branches += [Branch(f"Cc_{n}", n, 9, C=C_c) for n in (2, 3, 6, 7)]
coupled = Circuit(branches, ground=0)

Each grounded node gets its own ground symbol, placed outward from the center of the drawing:

In [ ]:
layout = {1: (-2, 1), 2: (-1, 1), 3: (-1, 0), 4: (-2, 0), 5: (1, 1), 6: (2, 1), 7: (2, 0), 8: (1, 0),
          9: (0, 1.6), 10: (0, -0.6)}
cq.draw_circuit_diagram(coupled, layout=layout, grounds="radial", label=False)

## Loops

`from_circuit` builds a loop basis from a spanning tree.
With `zero_capacitive=True` only the two superconducting rings keep a symbolic flux, named after the branch that closes each one.

In [ ]:
loops = Loops.from_circuit(coupled, zero_capacitive=True)

## Quantization

In [ ]:
result = quantize(coupled, loops, simplify=True)
assert result.coords.types == ("c", "c", "e", "e", "h", "h")
result

The four nondynamical coordinates are removed, leaving each qubit's $\theta$ (`c`), $\phi$ (`e`), and $\zeta$ (`h`).
The only coupling between the qubits is the charge term between the two compact modes:

In [ ]:
result.ham.E_C